In [1]:
import pandas as pd
import numpy as np

In [2]:
all_micr = pd.read_csv("../../CGMacros/microbes.csv")

In [3]:
matched_micr = pd.read_csv("microbes_sub_match.csv")

In [4]:
all_micr.shape, matched_micr.shape

((45, 1980), (45, 29))

In [13]:
all_micr.sum()

subject                                           1099.0
Abiotrophia defectiva                                5.0
Abiotrophia sp. HMSC24B09                            1.0
Acetivibrio ethanolgignens                           3.0
Acetivibrio ethanolgignens strain ACET-33324         6.0
                                                   ...  
[Ruminococcus] torques strain 2789STDY5608833       18.0
[Ruminococcus] torques strain 2789STDY5608867       16.0
[Ruminococcus] torques strain 2789STDY5834841       17.0
[Ruminococcus] torques strain 2789STDY5834889       17.0
bacterium LF-3                                      15.0
Length: 1980, dtype: float64

In [14]:
matched_micr.sum()

Unnamed: 0                                   990.0
Marvinbryantia formatexigens DSM 14469         3.0
Lachnospiraceae bacterium                     18.0
Oscillospiraceae bacterium VE202-24           18.0
Dorea sp. 5-2                                  3.0
Ruminococcaceae bacterium D16                 17.0
Fusicatenibacter saccharivorans               18.0
Lachnoclostridium                              1.0
Clostridia bacterium UC5.1-1D1                15.0
Prevotella                                     1.0
Anaerosporobacter mobilis DSM 15930            1.0
Ruminococcus champanellensis                  11.0
Prevotellaceae bacterium Marseille-P2826       3.0
Eubacterium                                    1.0
Ruminococcus sp. AT10                          7.0
Christensenella                                4.0
Bilophila sp. 4_1_30                          13.0
Parabacteroides                                1.0
Blautia                                        1.0
Muribaculum                    

In [91]:
id_cols = [c for c in all_micr.columns if c.lower() in {"subject", "sub", "participant", "id"}]
mic_cols = [c for c in all_micr.columns if c not in id_cols]

In [92]:
import re
def to_genus(col: str) -> str:
    return col.strip().split()[0]  # first word

def normalize_genus_token(tok: str) -> str:
    tok = tok.strip("[](){}") # normalize common wrappers: [Genus] -> Genus
    tok = re.sub(r"(_[A-Z]+)$", "", tok)      # Ruminococcus_A -> Ruminococcus  (GTDB-ish)
    tok = re.sub(r"^[^\w]+|[^\w]+$", "", tok) # trim weird punctuation
    return tok

In [93]:
genus_map = {c: normalize_genus_token(to_genus(c)) for c in mic_cols}

all_micr_genus = pd.concat(
    [
        all_micr[id_cols],
        all_micr[mic_cols]
          .rename(columns=genus_map)
          .groupby(axis=1, level=0)   # group same genus columns
          .sum()                      # or .mean(), depending on what "aggregation" means for you
    ],
    axis=1
)

In [94]:
all_micr_genus.shape

(45, 373)

In [95]:
all_micr_genus.describe()

,subject,Abiotrophia,Acetivibrio,Achromobacter,Acidaminococcus,Acidovorax,Acinetobacter,Actinobaculum,Actinomyces,Adlercreutzia,...,Valsa,Variovorax,Veillonella,Veillonellaceae,Watermelon,Weissella,Weizmannia,White,Zygosaccharomyces,bacterium
count,45.000000,45.000000,45.00000,45.000000,45.000000,45.000000,45.000000,45.000000,45.000000,45.000000,...,45.000000,45.000000,45.000000,45.000000,45.000000,45.000000,45.000000,45.000000,45.000000,45.000000
mean,24.422222,0.133333,0.20000,0.022222,0.533333,0.066667,0.444444,0.155556,4.555556,0.577778,...,0.177778,0.022222,2.044444,0.022222,0.022222,0.555556,0.022222,0.022222,0.022222,0.333333
std,14.627945,0.343776,0.40452,0.149071,1.120065,0.330289,0.502519,0.366529,5.025189,0.499495,...,0.386646,0.149071,2.513197,0.149071,0.149071,0.989847,0.149071,0.149071,0.149071,0.476731
min,1.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,12.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,2.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,23.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.000000,3.000000,1.000000,...,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
75%,36.000000,0.000000,0.00000,0.000000,0.000000,0.000000,1.000000,0.000000,5.000000,1.000000,...,0.000000,0.000000,3.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,1.000000
max,49.000000,1.000000,1.00000,1.000000,4.000000,2.000000,1.000000,1.000000,21.000000,1.000000,...,1.000000,1.000000,13.000000,1.000000,1.000000,4.000000,1.000000,1.000000,1.000000,1.000000


In [101]:
all_micr_genus.to_csv("microbes_genus.csv")

In [33]:
all_micr_genus.max()

subject                49.0
Abiotrophia             1.0
Acetivibrio             1.0
Achromobacter           1.0
Acidaminococcus         4.0
                       ... 
[Eubacterium            1.0
[Eubacterium]           8.0
[Propionibacterium]     1.0
[Ruminococcus]          4.0
bacterium               1.0
Length: 380, dtype: float64

In [35]:
from ete3 import NCBITaxa

ncbi = NCBITaxa(dbfile=r"C:\Users\varya\PycharmProjects\Nutrition\ete3_taxa.sqlite")
# If you haven't run it before on this machine, you may need:
ncbi.update_taxonomy_database()

NCBI database not present yet (first time used?)
Done. Parsing...


Loading node names...
2721626 names loaded.
428097 synonyms loaded.
Loading nodes...
2721626 nodes loaded.
Linking nodes...
Tree is loaded.
Updating database: C:\Users\varya\PycharmProjects\Nutrition\ete3_taxa.sqlite ...
 2721000 generating entries... 

Inserting synonyms:       5000 


Uploading to C:\Users\varya\PycharmProjects\Nutrition\ete3_taxa.sqlite



Inserting synonyms:      425000 

...

Loading node names...
2721626 names loaded.
428097 synonyms loaded.
Loading nodes...
2721626 nodes loaded.
Linking nodes...
Tree is loaded.
Updating database: C:\Users\varya\PycharmProjects\Nutrition\ete3_taxa.sqlite ...
 2721000 generating entries... 
Uploading to C:\Users\varya\PycharmProjects\Nutrition\ete3_taxa.sqlite


Inserting synonyms:      15000 

Inserting synonyms:      425000  

Inserting taxids:       10000  

Inserting taxids:       2720000     

In [37]:
def name_for_lookup(col: str) -> str:
    parts = col.strip().split()
    if not parts:
        return "unassigned"
    genus = normalize_genus_token(parts[0])

    # if "Genus sp. ..." then genus-only lookup is safer
    if len(parts) >= 2 and parts[1].lower() == "sp.":
        return genus

    # otherwise try "Genus species" first (often best), fall back to genus later
    if len(parts) >= 2:
        return f"{genus} {parts[1]}"
    return genus

In [38]:
# cache lookups so it doesn't hammer taxonomy calls
_tax_cache = {}

def rank_of(col: str, rank: str = "family") -> str:
    key = (col, rank)
    if key in _tax_cache:
        return _tax_cache[key]

    candidate = name_for_lookup(col)
    genus_only = candidate.split()[0] if candidate else "unassigned"

    def resolve(name: str):
        tr = ncbi.get_name_translator([name])
        if not tr:
            return None
        taxid = tr[name][0]
        lineage = ncbi.get_lineage(taxid)
        ranks = ncbi.get_rank(lineage)
        names = ncbi.get_taxid_translator(lineage)
        for tid in lineage[::-1]:
            if ranks.get(tid) == rank:
                return names.get(tid, "unassigned")
        return "unassigned"

    out = resolve(candidate)
    if out is None:
        out = resolve(genus_only)
    if out is None:
        out = "unassigned"

    _tax_cache[key] = out
    return out

In [98]:
# pick your rank: "family", "order", "class", "phylum"
RANK = "family"

rank_map = {c: rank_of(c, RANK) for c in mic_cols}

all_micr_by_rank = pd.concat(
    [
        all_micr[['subject']],
        all_micr[mic_cols]
          .rename(columns=rank_map)
          .groupby(axis=1, level=0)
          .sum()
    ],
    axis=1
)

In [102]:
all_micr_by_rank.to_csv("microbes_family.csv")

In [70]:
for k, v in _tax_cache:
    if k.strip().split()[0] == 'Ruminococcus':
        print(f"k: {k},v: {v}, family: {_tax_cache[k,v]}")

k: Ruminococcus bicirculans ,v: family, family: Oscillospiraceae
k: Ruminococcus callidus ATCC 27760 ,v: family, family: Oscillospiraceae
k: Ruminococcus champanellensis ,v: family, family: Oscillospiraceae
k: Ruminococcus faecis JCM 15917 ,v: family, family: Lachnospiraceae
k: Ruminococcus gauvreauii ,v: family, family: Oscillospiraceae
k: Ruminococcus gnavus ATCC 29149 ,v: family, family: Lachnospiraceae
k: Ruminococcus lactaris ,v: family, family: Lachnospiraceae
k: Ruminococcus lactaris ATCC 29176 ,v: family, family: Lachnospiraceae
k: Ruminococcus lactaris CC59_002D ,v: family, family: Lachnospiraceae
k: Ruminococcus sp. 5_1_39BFAA ,v: family, family: Oscillospiraceae
k: Ruminococcus sp. AT10 ,v: family, family: Oscillospiraceae
k: Ruminococcus sp. DSM 100440 ,v: family, family: Oscillospiraceae
k: Ruminococcus sp. JC304 ,v: family, family: Oscillospiraceae
k: Ruminococcus sp. JE7A12 ,v: family, family: Oscillospiraceae
k: Ruminococcus sp. Marseille-P3213 sp. Marseille-P3213 ,v: f

In [79]:
col = "Ruminococcus lactaris CC59_002D "
candidate = name_for_lookup(col)
genus_only = candidate.split()[0] if candidate else "unassigned"
candidate, genus_only

('Ruminococcus lactaris', 'Ruminococcus')

In [80]:
tr = ncbi.get_name_translator([candidate])
tr

{'Ruminococcus lactaris': [46228]}

In [82]:
taxid = tr[candidate][0]
lineage = ncbi.get_lineage(taxid)
ranks = ncbi.get_rank(lineage)
names = ncbi.get_taxid_translator(lineage)
lineage, ranks, names

([1, 131567, 2, 1783272, 1239, 186801, 3085636, 186803, 2316020, 46228],
 {1: 'no rank',
  2: 'domain',
  1239: 'phylum',
  46228: 'species',
  131567: 'cellular root',
  186801: 'class',
  186803: 'family',
  1783272: 'kingdom',
  2316020: 'genus',
  3085636: 'order'},
 {1: 'root',
  2: 'Bacteria',
  1239: 'Bacillota',
  46228: '[Ruminococcus] lactaris',
  131567: 'cellular organisms',
  186801: 'Clostridia',
  186803: 'Lachnospiraceae',
  1783272: 'Bacillati',
  2316020: 'Mediterraneibacter',
  3085636: 'Lachnospirales'})

In [84]:
all_micr_by_rank.max()

subject               49.0
Acetivibrionaceae      1.0
Acidaminococcaceae     6.0
Actinomycetaceae      22.0
Aerococcaceae          1.0
                      ... 
Vespertilionidae       1.0
Virgaviridae           1.0
Weeksellaceae          4.0
Yersiniaceae           1.0
unassigned            32.0
Length: 133, dtype: float64

In [108]:
# pick your rank: "family", "order", "class", "phylum"
RANK = "phylum"

rank_map = {c: rank_of(c, RANK) for c in mic_cols}

all_micr_by_rank = pd.concat(
    [
        all_micr[['subject']],
        all_micr[mic_cols]
          .rename(columns=rank_map)
          .groupby(axis=1, level=0)
          .sum()
    ],
    axis=1
)

In [109]:
all_micr_by_rank.to_csv("microbes_phylum.csv")